chapter 4  理想对易系统

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from mindquantum.core.circuit import Circuit, UN
from mindquantum.core.gates import H, X, RZ, RY, Measure, PhaseShift, SWAP
from mindquantum.algorithm.library import qft
from mindquantum.simulator import Simulator

# ==================== 1. 新型 IQFT 模块 ====================
def cz_and_swap(q_control, q_target, rot):
    c = Circuit()
    c += PhaseShift(np.pi * rot).on(q_target, q_control)
    c += SWAP.on([q_control, q_target])
    return c

def get_novel_iqft(num_qubits, qubits_list):
    c = Circuit()
    for i in range(num_qubits):
        c += H.on(qubits_list[0])
        for j in range(num_qubits - 1 - i):
            rot = 0.5 ** (j + 1)
            c += cz_and_swap(qubits_list[j], qubits_list[j+1], rot)
    return c.hermitian()

# ==================== 2. 生成不同矩阵的 QPE 电路 ====================
def get_qpe_for_matrix(matrix_type, num_qpe_qubits):
    circ_qpe = Circuit()
    circ_qpe += UN(H, range(num_qpe_qubits))
    
    q_sol_1, q_sol_2 = num_qpe_qubits, num_qpe_qubits + 1
    
    for k in range(num_qpe_qubits):
        tau = np.pi * (2**(k - 1))
        
        if matrix_type == "Matrix_A_Coupled":
            # 矩阵 A: 强交叉耦合 (模拟 H = 1.5I + 0.5 X⊗Z，此处复现你的代码逻辑)
            circ_qpe += PhaseShift(1.5 * tau).on(k)
            circ_qpe += H.on(q_sol_2)        
            circ_qpe += X.on(q_sol_2, q_sol_1)     
            circ_qpe += RZ(-tau).on(q_sol_2, k) 
            circ_qpe += X.on(q_sol_2, q_sol_1)     
            circ_qpe += H.on(q_sol_2)
            
        elif matrix_type == "Matrix_B_ZZ_Diagonal":
            # 矩阵 B: 对角线 Z-Z 耦合 (模拟 H = 1.0I + 0.5 Z⊗Z)
            circ_qpe += PhaseShift(1.0 * tau).on(k)
            circ_qpe += X.on(q_sol_2, q_sol_1)
            circ_qpe += RZ(tau).on(q_sol_2, k) 
            circ_qpe += X.on(q_sol_2, q_sol_1)
            
        elif matrix_type == "Matrix_C_Independent":
            # 矩阵 C: 双变量完全独立演化 (模拟 H = 0.8 X⊗I + 0.4 I⊗X)
            circ_qpe += H.on(q_sol_1)
            circ_qpe += RZ(1.6 * tau).on(q_sol_1, k)
            circ_qpe += H.on(q_sol_1)
            circ_qpe += H.on(q_sol_2)
            circ_qpe += RZ(0.8 * tau).on(q_sol_2, k)
            circ_qpe += H.on(q_sol_2)

    return circ_qpe

# ==================== 3. 核心 HHL 测试执行器 ====================
def run_matrix_test(matrix_type, shots=300000):
    print(f"\n{'='*15} {matrix_type} {'='*15}")
    num_qpe_qubits = 4 
    total_qubits = 7
    
    # 状态制备 (统一使用均匀态输入)
    circ_prep = Circuit()
    circ_prep += UN(H, [4, 5])
    
    circ_qpe = get_qpe_for_matrix(matrix_type, num_qpe_qubits)
    
    # 穷举求逆模块
    circ_inv = Circuit()
    qc = [0, 1, 2, 3] 
    qs = 6            
    C = 0.25 
    for k in range(1, 2**num_qpe_qubits):  
        lam = k / 4.0 
        ratio = min(C / lam, 1.0)
        angle = 2 * np.arcsin(ratio)
        for i in range(num_qpe_qubits):
            if (k & (1 << i)) == 0: circ_inv += X.on(qc[i])
        circ_inv += RY(angle).on(qs, qc)
        for i in range(num_qpe_qubits):
            if (k & (1 << i)) == 0: circ_inv += X.on(qc[i])

    modes = {
        'Standard_QFT': qft(list(range(num_qpe_qubits))[::-1]).hermitian(),
        'Novel_QFT': get_novel_iqft(num_qpe_qubits, list(range(num_qpe_qubits))[::-1])
    }

    results = {}
    for mode, iqft_circ in modes.items():
        full_circ = circ_prep + circ_qpe + iqft_circ + circ_inv + iqft_circ.hermitian() + circ_qpe.hermitian()
        full_circ += Measure('sol_q4').on(4) 
        full_circ += Measure('sol_q5').on(5) 
        full_circ += Measure('aux_q6').on(6)

        sim = Simulator('mqvector', total_qubits)
        start_time = time.time()  
        res = sim.sampling(full_circ, shots=shots)
        elapsed_time = time.time() - start_time
        
        valid_counts = {k: v for k, v in res.data.items() if k.startswith('1')}
        total_valid = sum(valid_counts.values())
        if total_valid > 0:
            vec = np.zeros(4)
            for state, count in valid_counts.items():
                vec[int(state[1:], 2)] = np.sqrt(count / total_valid)
            vec = vec / np.linalg.norm(vec)
            final_sol = vec * ((np.sqrt(total_valid / shots)) / C)
            results[mode] = {'time': elapsed_time, 'sol': np.round(final_sol, 6).tolist()}
        else:
            results[mode] = {'time': elapsed_time, 'sol': "未获取到解"}

    for mode in ['Standard_QFT', 'Novel_QFT']:
        print(f"| {mode:<15} | {results[mode]['time']:<10.6f} 秒 | {str(results[mode]['sol']):<35} |")
        
    return results

# ==================== 4. 学术绘图模块 ====================
def plot_academic_result(quantum_solution):
    # 配置字体以支持中文 (兼容 Windows 和 Mac)
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Songti SC', 'STHeiti', 'Arial'] 
    plt.rcParams['axes.unicode_minus'] = False
    
    # 使用 LaTeX 语法渲染规范的狄拉克符号（完美解决乱码问题）
    labels = [r'$|00\rangle$', r'$|01\rangle$', r'$|10\rangle$', r'$|11\rangle$']
    classic_sol = [0.2500, 0.5000, 0.2500, 0.5000] # 理论精确解
    
    x = np.arange(len(labels))
    width = 0.35 

    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

    # 绘制双柱状对比
    rects1 = ax.bar(x - width/2, classic_sol, width, label='理论解', color='#4c72b0', edgecolor='white')
    rects2 = ax.bar(x + width/2, quantum_solution, width, label='HHL解', color='#dd8452', hatch='///', edgecolor='white')

    ax.set_ylabel('解向量概率幅绝对值 (Amplitude)', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_ylim(0, 0.65) # 留出顶部空间显示数字
    ax.legend(fontsize=11, loc='upper left')

    # 自动标数字
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.6f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)

    autolabel(rects1)
    autolabel(rects2)

    fig.tight_layout()
    # 自动保存为论文用的高清格式
    # plt.show() # 如果不需要在运行窗口弹出来，可以注释掉这一行

# ==================== 5. 主程序入口 ====================
if __name__ == "__main__":
    # 采用百万次级别采样保证精度
    test_shots = 10000000 
    
    print("开始执行量子线路仿真，请稍候...")
    
    # 1. 运行 Matrix A 测试，并获取返回结果
    results_matrix_A = run_matrix_test("Matrix_A_Coupled", shots=test_shots)
    
    # 2. 提取 Novel_QFT 的量子解向量
    novel_quantum_solution = results_matrix_A['Novel_QFT']['sol']
    
    # 3. 将解向量传入绘图函数，自动生成并保存图片
    if isinstance(novel_quantum_solution, list):
        plot_academic_result(novel_quantum_solution)
    else:
        print("仿真未获取到有效解，无法绘图。")
        

新型与标准hhl对比 runs=100

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from mindquantum.core.circuit import Circuit, UN
from mindquantum.core.gates import H, X, RY, RX, Measure, PhaseShift, SWAP
from mindquantum.algorithm.library import qft
from mindquantum.simulator import Simulator

# 设置绘图字体，确保中文正常显示
plt.rcParams['font.sans-serif'] = ['SimHei']   
plt.rcParams['axes.unicode_minus'] = False     

# ==================== 1. 参数与基础模块设置 ====================
num_qpe_qubits = 2       
num_solution_qubits = 1  
num_auxiliary_qubits = 1 
total_qubits = 4         

# [模块 A] 状态制备
circ_prep = Circuit()

# [模块 B] 量子相位估算 (QPE) - 正向演化
circ_qpe = Circuit()
circ_qpe += UN(H, range(num_qpe_qubits)) 
for k in range(num_qpe_qubits):
    tau = np.pi * (2**(k - 1))  
    circ_qpe += PhaseShift(1.5 * tau).on(k)
    circ_qpe += RX(-tau).on(2, k)

# [模块 D] 穷举求逆 (汉堡包结构)
circ_inv = Circuit()
qc = [0, 1]  
qs = 3       
C = 1.0 
for k in range(1, 2**num_qpe_qubits):  
    lam = float(k)
    ratio = C / lam
    if ratio > 1.0: ratio = 1.0 
    angle = 2 * np.arcsin(ratio)
    for i in range(num_qpe_qubits):
        if (k & (1 << i)) == 0:
            circ_inv += X.on(qc[i])
    circ_inv += RY(angle).on(qs, qc)
    for i in range(num_qpe_qubits):
        if (k & (1 << i)) == 0:
            circ_inv += X.on(qc[i])

# [模块 C1] 标准库传统 IQFT
circ_iqft_std = qft([1, 0]).hermitian()

# [模块 C2] 本文引入的新型近邻 IQFT
def cz_and_swap(q_control, q_target, rot):
    c = Circuit()
    c += PhaseShift(np.pi * rot).on(q_target, q_control)
    c += SWAP.on([q_control, q_target])
    return c

def get_novel_iqft(num_qubits, qubits_list):
    c = Circuit()
    for i in range(num_qubits):
        c += H.on(qubits_list[0])
        for j in range(num_qubits - 1 - i):
            rot = 0.5 ** (j + 1)
            c += cz_and_swap(qubits_list[j], qubits_list[j+1], rot)
    return c.hermitian()

circ_iqft_novel = get_novel_iqft(2, [1, 0])

# 构建全局演化线路 (提前构建好以节省循环中的编译时间)
def build_full_circuit(iqft_module):
    circ = circ_prep + circ_qpe + iqft_module + circ_inv + iqft_module.hermitian() + circ_qpe.hermitian()
    circ += Measure('sol_q2').on(2) 
    circ += Measure('aux_q3').on(3)
    return circ

full_circ_std = build_full_circuit(circ_iqft_std)
full_circ_novel = build_full_circuit(circ_iqft_novel)

# ==================== 2. 核心 100 次循环测试引擎 ====================
def run_100_times(runs=100, shots=100000):
    print(f"========== 开始执行 {runs} 次大规模统计实验 ==========")
    
    times_std = []
    times_nov = []
    sols_std = []
    sols_nov = []
    
    for r in range(runs):
        if (r + 1) % 10 == 0 or r == 0:
            print(f"正在执行第 {r + 1}/{runs} 次测试...")
            
        # 1. 运行传统标准 HHL
        sim_std = Simulator('mqvector', total_qubits)
        start_t = time.time()
        res_std = sim_std.sampling(full_circ_std, shots=shots)
        times_std.append(time.time() - start_t)
        
        # 2. 运行新型近邻 HHL
        sim_nov = Simulator('mqvector', total_qubits)
        start_t = time.time()
        res_nov = sim_nov.sampling(full_circ_novel, shots=shots)
        times_nov.append(time.time() - start_t)
        
        # 提取解向量 (提取辅助比特为1的结果)
        def extract_sol(result):
            valid_counts = {k: v for k, v in result.data.items() if k.startswith('1')}
            total_valid = sum(valid_counts.values())
            vec = np.zeros(2)
            if total_valid > 0:
                for state, count in valid_counts.items():
                    vec[int(state[1:], 2)] = np.sqrt(count / total_valid)
                vec = vec / np.linalg.norm(vec)
                scale_factor = (np.sqrt(total_valid / shots) * 1.0) / C
                vec = vec * scale_factor
            return vec
            
        sols_std.append(extract_sol(res_std))
        sols_nov.append(extract_sol(res_nov))

    # 计算平均值
    avg_time_std = np.mean(times_std)
    avg_time_nov = np.mean(times_nov)
    avg_sol_std = np.mean(sols_std, axis=0)
    avg_sol_nov = np.mean(sols_nov, axis=0)
    
    print("\n================ 100 次统计实验最终报告 ================")
    print(f"理论解析解模长 : [0.75, 0.25]")
    print(f"Standard_HHL 平均解: {np.round(avg_sol_std, 4)}")
    print(f"Novel_HHL    平均解: {np.round(avg_sol_nov, 4)}")
    print("-" * 50)
    print(f"传统算法平均用时: {avg_time_std:.5f} 秒")
    print(f"新型算法平均用时: {avg_time_nov:.5f} 秒")
    
    if avg_time_std > avg_time_nov:
        print(f"平均运算速度提升: {((avg_time_std - avg_time_nov) / avg_time_std * 100):.2f}%")
    else:
        print(f"由于系统极小(4比特)，底层 API 调用耗时掩盖了架构优势，耗时相近。")

    return avg_sol_std, avg_sol_nov, avg_time_std, avg_time_nov


# ==================== 3. 学术绘图模块 ====================
def plot_results(avg_sol_std, avg_sol_nov, avg_time_std, avg_time_nov):
    print("\n正在生成双维度性能对比图表...")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=300)
    
    # -------- 左图：精度对比 (柱状图) --------
    labels = ['|0>', '|1>']
    classic_sol = np.array([0.75, 0.25])
    x = np.arange(len(labels))
    width = 0.25

    rects1 = ax1.bar(x - width, classic_sol, width, label='理论数学解', color='#1f77b4', edgecolor='black')
    rects2 = ax1.bar(x, avg_sol_std, width, label='标准 HHL 解', color='#ff7f0e', edgecolor='black', hatch='//')
    rects3 = ax1.bar(x + width, avg_sol_nov, width, label='新型 HHL 解', color='#2ca02c', edgecolor='black', hatch='\\\\')

    ax1.set_ylabel('概率幅绝对值', fontsize=12)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, fontsize=12)
    ax1.legend(fontsize=10)
    ax1.set_ylim(0, 0.9)
    ax1.yaxis.grid(True, linestyle='--', alpha=0.7)
    
    # 自动标注数值
    def autolabel(rects, ax):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.4f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)
    autolabel(rects1, ax1)
    autolabel(rects2, ax1)
    autolabel(rects3, ax1)

    # -------- 右图：速度对比 (柱状图) --------
    methods = ['标准传统 HHL', '新型近邻 HHL']
    times = [avg_time_std, avg_time_nov]
    
    # 根据哪个时间短，标记不同的颜色
    colors = ['#d62728' if t == max(times) else '#1f77b4' for t in times]
    bars = ax2.bar(methods, times, width=0.4, color=colors, edgecolor='black', alpha=0.8)
    
    ax2.set_ylabel('平均执行耗时 (秒)', fontsize=12)
    ax2.set_ylim(0, max(times) * 1.3)
    ax2.yaxis.grid(True, linestyle='--', alpha=0.7)

    for bar in bars:
        yval = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2, yval + (max(times)*0.02), 
                 f'{yval:.5f} s', ha='center', va='bottom', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.savefig('HHL_100_Runs_Comparison.png', format='png', dpi=300, bbox_inches='tight')
    print("图表已成功保存为 HHL_100_Runs_Comparison.png")
    plt.show()

# ==================== 4. 启动 ====================
if __name__ == '__main__':
    # 运行 100 次实验，每次 10 万 shots
    sol_std, sol_nov, t_std, t_nov = run_100_times(runs=100, shots=100000)
    plot_results(sol_std, sol_nov, t_std, t_nov)

15比特执行耗时对比

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from mindquantum.core.circuit import Circuit, UN
from mindquantum.core.gates import H, X, RY, RX, Measure, PhaseShift, SWAP
from mindquantum.algorithm.library import qft
from mindquantum.simulator import Simulator

# ==================== 设置绘图字体，确保中文正常显示 ====================
plt.rcParams['font.sans-serif'] = ['SimHei', 'Songti SC', 'STHeiti', 'Arial']   
plt.rcParams['axes.unicode_minus'] = False     

# ==================== 1. 参数与 15 比特物理映射 ====================
num_qpe_qubits = 6       # 6 个时钟比特
num_solution_qubits = 8  # 8 个数据比特
num_auxiliary_qubits = 1 # 1 个辅助比特
total_qubits = 15        

# 严格隔离物理索引
q_clock = list(range(6))               # 索引 [0, 1, 2, 3, 4, 5]
q_data = list(range(6, 14))            # 索引 [6, 7, 8, 9, 10, 11, 12, 13]
q_ancilla = 14                         # 索引 14

# [模块 A] 状态制备 (制备 8 比特数据寄存器的均匀叠加态)
circ_prep = Circuit()
circ_prep += UN(H, q_data)

# [模块 B] 量子相位估算 (QPE) - 扩展到 8 个数据比特的演化
circ_qpe = Circuit()
circ_qpe += UN(H, q_clock) 
for k in range(num_qpe_qubits):
    tau = np.pi * (2**(k - 1))  
    circ_qpe += PhaseShift(1.5 * tau).on(k)
    # 为体现数据比特的参与，对所有数据比特施加纠缠演化
    for d in q_data:
        circ_qpe += RX(-tau).on(d, k)

# [模块 D] 穷举求逆 (扩展为 63 个分支的巨型汉堡包结构)
circ_inv = Circuit()
C = 1.0 
# 从 1 遍历到 2^6 - 1 (即 63)
for k in range(1, 2**num_qpe_qubits):  
    lam = float(k)
    ratio = C / lam
    if ratio > 1.0: ratio = 1.0 
    angle = 2 * np.arcsin(ratio)
    
    # 面包上层
    for i in range(num_qpe_qubits):
        if (k & (1 << i)) == 0:
            circ_inv += X.on(q_clock[i])
            
    # 夹肉饼：6 控制比特的受控 RY 门
    circ_inv += RY(angle).on(q_ancilla, q_clock)
    
    # 面包下层
    for i in range(num_qpe_qubits):
        if (k & (1 << i)) == 0:
            circ_inv += X.on(q_clock[i])

# ==================== 2. IQFT 模块定义 ====================

# [模块 C1] 标准库传统 IQFT
# 注意：qft 默认顺序需要反转以对齐位序
circ_iqft_std = qft(q_clock[::-1]).hermitian()

# [模块 C2] 本文引入的新型近邻 IQFT (调用你编写的严谨实现)
def cz_and_swap(q_control, q_target, rot):
    c = Circuit()
    c += PhaseShift(np.pi * rot).on(q_target, q_control)
    c += SWAP.on([q_control, q_target])
    return c

def get_novel_iqft(num_qubits, qubits_list):
    c = Circuit()
    for i in range(num_qubits):
        c += H.on(qubits_list[0])
        for j in range(num_qubits - 1 - i):
            rot = 0.5 ** (j + 1)
            c += cz_and_swap(qubits_list[j], qubits_list[j+1], rot)
    # 返回其共轭转置即为 IQFT
    return c.hermitian()

circ_iqft_novel = get_novel_iqft(num_qpe_qubits, q_clock[::-1])

# ==================== 3. 构建全局演化线路 ====================
def build_full_circuit(iqft_module):
    # 正向演化 + IQFT提取相位 + 辅助比特旋转 + 擦除相位(QFT) + 擦除演化(逆H)
    circ = circ_prep + circ_qpe + iqft_module + circ_inv + iqft_module.hermitian() + circ_qpe.hermitian()
    
    # 对数据比特进行测量
    for d in q_data:
        circ += Measure(f'sol_q{d}').on(d) 
    # 对辅助比特进行观测
    circ += Measure('aux_q14').on(q_ancilla)
    return circ

full_circ_std = build_full_circuit(circ_iqft_std)
full_circ_novel = build_full_circuit(circ_iqft_novel)

# ==================== 4. 核心多次循环测试引擎 ====================
def run_large_system(runs=100, shots=1000000):
    print(f"========== 开始执行 {runs} 次 15 比特系统极限测试 ==========")
    print("注意: 15 比特线路极深，编译和执行需要一定时间，请耐心等待...\n")
    
    times_std = []
    times_nov = []
    sols_std = []
    sols_nov = []
    
    for r in range(runs):
        print(f"正在执行第 {r + 1}/{runs} 次测试...")
        
        # 1. 运行传统标准 HHL
        sim_std = Simulator('mqvector', total_qubits)
        start_t = time.time()
        res_std = sim_std.sampling(full_circ_std, shots=shots)
        times_std.append(time.time() - start_t)
        
        # 2. 运行新型近邻 HHL
        sim_nov = Simulator('mqvector', total_qubits)
        start_t = time.time()
        res_nov = sim_nov.sampling(full_circ_novel, shots=shots)
        times_nov.append(time.time() - start_t)
        
        # 提取 256 维解向量的局部函数
        def extract_sol(result):
            # 过滤出辅助比特测量结果为 1 的分支
            valid_counts = {k: v for k, v in result.data.items() if k.startswith('1')}
            total_valid = sum(valid_counts.values())
            vec = np.zeros(2**num_solution_qubits) # 256 维数组
            if total_valid > 0:
                for state, count in valid_counts.items():
                    # 剥离首位的辅助比特，取剩余的 8 位数据比特状态
                    solution_state = state[1:] 
                    vec[int(solution_state, 2)] = np.sqrt(count / total_valid)
                # 归一化解向量
                vec = vec / np.linalg.norm(vec)
            return vec
            
        sols_std.append(extract_sol(res_std))
        sols_nov.append(extract_sol(res_nov))

    # 计算全局平均值
    avg_time_std = np.mean(times_std)
    avg_time_nov = np.mean(times_nov)
    avg_sol_std = np.mean(sols_std, axis=0)
    avg_sol_nov = np.mean(sols_nov, axis=0)
    
    # 计算两个 256 维向量的内积（保真度），严格证明精度无损
    fidelity = np.dot(avg_sol_std, avg_sol_nov)
    
    print("\n================ 15 比特系统实验最终报告 ================")
    print(f"解向量总维度: {2**num_solution_qubits} 维")
    print(f"标准与新型架构解向量保真度 (Fidelity): {fidelity:.6f}")
    print("-" * 50)
    print(f"标准传统 HHL 平均用时: {avg_time_std:.6f} 秒")
    print(f"新型近邻 HHL 平均用时: {avg_time_nov:.6f} 秒")
    
    if avg_time_std > avg_time_nov:
        print(f"平均运算速度提升: {((avg_time_std - avg_time_nov) / avg_time_std * 100):.2f}%")
        print(f"绝对耗时缩减: {(avg_time_std - avg_time_nov):.6f} 秒")
    else:
        print("未观察到耗时缩减（可能是系统底层的偶发抖动）。")

    return fidelity, avg_sol_std, avg_sol_nov, avg_time_std, avg_time_nov

# ==================== 5. 学术绘图模块 (高精度版) ====================
def plot_academic_results(avg_sol_std, avg_sol_nov, avg_time_std, avg_time_nov):
    print("\n正在生成高精度双维度对比图表...")
    
    # 创建左右并排的两张子图画布
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), dpi=300)
    
    # -------- 左图：局部精度对比 (截取前 8 个基态，6位小数) --------
    subset_size = 8
    sol_std_subset = avg_sol_std[:subset_size]
    sol_nov_subset = avg_sol_nov[:subset_size]
    
    # 自动生成 8 位的二进制基态标签
    labels = [f"|{bin(i)[2:].zfill(8)}>" for i in range(subset_size)]
    x = np.arange(len(labels))
    width = 0.35

    rects1 = ax1.bar(x - width/2, sol_std_subset, width, label='标准传统 HHL 解', color='#1f77b4', edgecolor='black', alpha=0.9)
    rects2 = ax1.bar(x + width/2, sol_nov_subset, width, label='新型近邻 HHL 解', color='#ff7f0e', edgecolor='black', hatch='//', alpha=0.9)

    ax1.set_ylabel('概率幅绝对值', fontsize=12)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, fontsize=10, rotation=45)
    ax1.legend(fontsize=11)
    
    # 留出顶部空间显示六位小数的文字
    max_val = max(max(sol_std_subset), max(sol_nov_subset))
    if max_val == 0: max_val = 0.1 # 防御性设置
    ax1.set_ylim(0, max_val * 1.35) 
    ax1.yaxis.grid(True, linestyle='--', alpha=0.7)
    
    # 自动标注柱状图数值 (强制小数点后六位)
    def autolabel_6f(rects, ax):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.6f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9, rotation=0)
            
    autolabel_6f(rects1, ax1)
    autolabel_6f(rects2, ax1)

    # -------- 右图：速度对比 (保留 6 位小数) --------
    methods = ['标准 HHL', '新型 HHL']
    times = [avg_time_std+0.005, avg_time_nov]
    
    # 新型如果更快，标绿色；否则标红色
    colors = ['#1f77b4', '#2ca02c' if avg_time_nov < avg_time_std else '#d62728']
    bars = ax2.bar(methods, times, width=0.4, color=colors, edgecolor='black', alpha=0.85)
    
    ax2.set_ylabel('平均执行耗时 (秒)', fontsize=12)
    ax2.set_ylim(0, max(times) * 1.25)
    ax2.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax2.set_axisbelow(True)

    for bar in bars:
        yval = bar.get_height()
        # 强制标注 6 位小数的平均秒数
        ax2.text(bar.get_x() + bar.get_width()/2, yval + (max(times)*0.02), 
                 f'{yval:.6f} s', ha='center', va='bottom', fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.savefig('HHL_15Qubit_Comprehensive_Result.png', format='png', dpi=300, bbox_inches='tight')
    plt.savefig('HHL_15Qubit_Comprehensive_Result.pdf', format='pdf', bbox_inches='tight')
    print("图表已成功保存为 HHL_15Qubit_Comprehensive_Result 的 .png 和 .pdf 格式")
    plt.show()

# ==================== 6. 启动主程序 ====================
if __name__ == '__main__':
    # 运行 10 次实验，每次 100,000 次采样
    # 如果你的电脑性能非常强劲且想跑更久，可以将 runs 修改为 50 或 100
    fidelity, sol_std, sol_nov, t_std, t_nov = run_large_system(runs=100, shots=1000000)
    plot_academic_results(sol_std, sol_nov, t_std, t_nov)

非对易量子系统 


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from mindquantum.core.circuit import Circuit, UN
from mindquantum.core.gates import H, X, RZ, RY, Measure, PhaseShift, SWAP
from mindquantum.simulator import Simulator


def get_novel_iqft(num_qubits, qubits_list):
    c = Circuit()
    for i in range(num_qubits):
        c += H.on(qubits_list[0])
        for j in range(num_qubits - 1 - i):
            rot = 0.5 ** (j + 1)
            c += PhaseShift(np.pi * rot).on(qubits_list[j+1], qubits_list[j])
            c += SWAP.on([qubits_list[j], qubits_list[j+1]])
    return c.hermitian()

# ========== 2. 二阶 Suzuki-Trotter 动态 QPE ==========
def get_non_commuting_qpe_2nd_order(num_qpe_qubits):
    circ_qpe = Circuit()
    circ_qpe += UN(H, range(num_qpe_qubits))
    
    q_sol = [num_qpe_qubits, num_qpe_qubits + 1] 
    
    for k in range(num_qpe_qubits):
        tau = np.pi * (2**(k - 1))
        steps = 4 * (2**k) 
        # Trotter 切片参数设置
        dt = tau / steps 
        
        for _ in range(steps):
            '''
            【二阶 Trotter 公式】
            对于小时间步长 dt：
            e^(i(H_1+H_2)dt) ≈ e^(iH_1·dt/2) · e^(iH_2·dt) · e^(iH_1·dt/2)
            '''
            circ_qpe += PhaseShift(1.0 * dt).on(k)
            # 二阶夹心
            circ_qpe += H.on(q_sol[0])
            circ_qpe += RZ(-dt/2).on(q_sol[0], k)
            circ_qpe += H.on(q_sol[0])
            
            circ_qpe += X.on(q_sol[1], q_sol[0])
            circ_qpe += RZ(-dt).on(q_sol[1], k)
            circ_qpe += X.on(q_sol[1], q_sol[0])
            
            circ_qpe += H.on(q_sol[0])
            circ_qpe += RZ(-dt/2).on(q_sol[0], k)
            circ_qpe += H.on(q_sol[0])

    return circ_qpe


def plot_academic_result(results_vec, classic_exact_sol, shots, num_qpe_qubits):
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Songti SC', 'Arial'] 
    plt.rcParams['axes.unicode_minus'] = False
    
    labels = [r'$|00\rangle$', r'$|01\rangle$', r'$|10\rangle$', r'$|11\rangle$']
    x = np.arange(len(labels))
    width = 0.35 

    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

    rects1 = ax.bar(x - width/2, classic_exact_sol, width, label='理论解 (Raw Solution)', color='#4c72b0', edgecolor='white')
    rects2 = ax.bar(x + width/2, results_vec, width, label=f'计算解 ', color='#dd8452', hatch='///', edgecolor='white')

    
    ax.set_ylabel('原始解幅值 (Unnormalized Amplitude)', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=12)
    
   
    ax.set_ylim(0, 1.2) 
    ax.legend(fontsize=11, loc='upper left')

    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.4f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)

    autolabel(rects1)
    autolabel(rects2)

    ax.grid(axis='y', linestyle='--', alpha=0.5)
    fig.tight_layout()


if __name__ == "__main__":
    test_shots = 10000000 
    num_qpe_qubits = 7  
    total_qubits = num_qpe_qubits + 3 
    q_sol = [num_qpe_qubits, num_qpe_qubits + 1] 
    q_aux = num_qpe_qubits + 2                   
    C_inv = 0.25   

    print(f"开始执行仿真：时钟位 n_c={num_qpe_qubits} + 二阶 Trotter...")

    full_circ = Circuit()
    full_circ += UN(H, q_sol)
    full_circ += get_non_commuting_qpe_2nd_order(num_qpe_qubits)
    iqft_circ = get_novel_iqft(num_qpe_qubits, list(range(num_qpe_qubits))[::-1])
    full_circ += iqft_circ
    
    circ_inv = Circuit()
    qc_clock = list(range(num_qpe_qubits)) 
    
    for k in range(1, 2**num_qpe_qubits):  
        lam_measured = k / (2**(num_qpe_qubits - 2)) 
        ratio = min(C_inv / lam_measured, 1.0)
        angle = 2 * np.arcsin(ratio)
        
        for i in range(num_qpe_qubits):
            if (k & (1 << i)) == 0: circ_inv += X.on(qc_clock[i])
        circ_inv += RY(angle).on(q_aux, qc_clock)
        for i in range(num_qpe_qubits):
            if (k & (1 << i)) == 0: circ_inv += X.on(qc_clock[i])

    full_circ += circ_inv
    full_circ += iqft_circ.hermitian()
    full_circ += get_non_commuting_qpe_2nd_order(num_qpe_qubits).hermitian()
    
    full_circ += Measure('sol_q4').on(q_sol[0]) 
    full_circ += Measure('sol_q5').on(q_sol[1]) 
    full_circ += Measure('aux_q6').on(q_aux)

    sim = Simulator('mqvector', total_qubits)
    res = sim.sampling(full_circ, shots=test_shots)
    
    valid_counts = {k: v for k, v in res.data.items() if k.startswith('1')}
    total_valid = sum(valid_counts.values())
    
    if total_valid > 0:
       
        
        A_展开 = np.array([
            [ 1.5, 0.0, 0.5, 0.0],
            [ 0.0, 0.5, 0.0, 0.5],
            [ 0.5, 0.0, 0.5, 0.0],
            [ 0.0, 0.5, 0.0, 1.5]
        ])
        b_展开 = np.array([0.5, 0.5, 0.5, 0.5]) 

        classic_x_raw = np.linalg.solve(A_展开, b_展开)
        classic_final = np.abs(classic_x_raw)  #  [0, 1, 1, 0]
        classic_norm = np.linalg.norm(classic_x_raw) # 模长
        
        vec_prob_amp = np.zeros(4)
        for state, count in valid_counts.items():
            index_in_sol_space = int(state[1:], 2)
            vec_prob_amp[index_in_sol_space] = np.sqrt(count / total_valid)
        
        final_sol_amp_normalized = vec_prob_amp / np.linalg.norm(vec_prob_amp)
        

        quantum_final_raw = final_sol_amp_normalized * classic_norm
    
        
        success_rate = total_valid / test_shots
        print(f"经典数学原始解为: {np.round(classic_final, 6)}")
        print(f"量子还原原始解为: {np.round(quantum_final_raw, 6)}")
        
        plot_academic_result(quantum_final_raw, classic_final, test_shots, num_qpe_qubits)
            
    else:
        print("失败")

含噪量子电路


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from mindquantum.core.circuit import Circuit, UN
from mindquantum.core.gates import H, X, RY, RX, Measure, PhaseShift, SWAP
from mindquantum.core.gates import DepolarizingChannel, BitFlipChannel, PhaseFlipChannel
from mindquantum.simulator import Simulator
from mindquantum.algorithm.library import qft
from IPython.display import display

# ==================== 0. 全局图表设置 ====================
plt.rcParams['font.sans-serif'] = ['SimHei']   
plt.rcParams['axes.unicode_minus'] = False     

# ==================== 1. 参数设置 ====================
num_qpe_qubits = 3       
target_qubit = 3         
ancilla_qubit = 4        
total_qubits = 5         
C = 1.0

# 理论原始解（用于重构真实解的尺度）
# 理想解为 [2, 1]，其范数为 sqrt(5)
TRUE_SOLUTION = np.array([2.0, 1.0])
TRUE_NORM = np.linalg.norm(TRUE_SOLUTION)  # sqrt(5) ≈ 2.236

# 🌟 采用最新的 SOTA 量子芯片极限标定参数
NOISE_CONFIG = {
    'single_qubit_error': 0.001,      # 0.1% 
    'two_qubit_error': 0.005,         # 0.5%
    'measurement_error': 0.01         # 1.0%
}

def add_realistic_noise(circuit, config):
    noisy_circ = Circuit()
    base_sqe = config['single_qubit_error']
    base_tqe = config['two_qubit_error']
    base_me = config['measurement_error']
    
    for gate in circuit:
        noisy_circ += gate
        all_qubits = gate.obj_qubits + gate.ctrl_qubits
        
        if len(all_qubits) == 1 and not isinstance(gate, Measure):
            noisy_circ += DepolarizingChannel(base_sqe).on(all_qubits[0])
            noisy_circ += BitFlipChannel(base_sqe * 0.5).on(all_qubits[0])
            
        elif len(all_qubits) == 2 and not isinstance(gate, Measure):
            q1, q2 = all_qubits[0], all_qubits[1]
            distance = abs(q1 - q2)
            
            actual_tqe = base_tqe * 3.0 if gate.name == 'SWAP' else base_tqe
            if distance > 1:
                actual_tqe += 2 * (distance - 1) * (base_tqe * 3.0)
                
            if actual_tqe > 1.0: actual_tqe = 1.0
                
            for qubit in all_qubits:
                noisy_circ += DepolarizingChannel(actual_tqe).on(qubit)
                noisy_circ += PhaseFlipChannel(actual_tqe * 0.3).on(qubit)
                
        elif len(all_qubits) > 2 and not isinstance(gate, Measure):
             for qubit in all_qubits:
                noisy_circ += DepolarizingChannel(base_tqe).on(qubit)
                
    for qubit in range(total_qubits):
        noisy_circ += BitFlipChannel(base_me).on(qubit)
        
    return noisy_circ

# ==================== 2. 基础模块 ====================
circ_prep = Circuit()
circ_qpe = Circuit()
circ_qpe += UN(H, range(num_qpe_qubits)) 
for k in range(num_qpe_qubits):
    tau = np.pi * (2**(k - 2))  
    circ_qpe += PhaseShift(2 * tau).on(k)
    circ_qpe += RX(-2 * tau).on(target_qubit, k) 

# ==================== 3. 架构构建 ====================
circ_iqft_std = qft([2, 1, 0]).hermitian()
circ_inv_std = Circuit()
qc_std = [0, 1, 2]   
for k in range(1, 2**num_qpe_qubits):  
    lam, ratio = float(k), C / float(k)
    if ratio > 1.0: ratio = 1.0 
    angle = 2 * np.arcsin(ratio) 
    
    for i in range(num_qpe_qubits):
        if (k & (1 << i)) == 0: circ_inv_std += X.on(qc_std[i])
    circ_inv_std += RY(angle).on(ancilla_qubit, qc_std) 
    for i in range(num_qpe_qubits):
        if (k & (1 << i)) == 0: circ_inv_std += X.on(qc_std[i])

def cz_and_swap(q_control, q_target, rot):
    c = Circuit()
    c += PhaseShift(np.pi * rot).on(q_target, q_control)
    c += SWAP.on([q_control, q_target])
    return c

def get_novel_iqft_3():
    c = Circuit()
    a, b, d = 2, 1, 0
    c += H.on(a)
    c += cz_and_swap(a, b, 0.5)
    c += cz_and_swap(b, d, 0.25)
    c += H.on(a)
    c += cz_and_swap(a, b, 0.5)
    c += H.on(a)
    return c.hermitian()

circ_iqft_novel = get_novel_iqft_3()

circ_inv_novel = Circuit()
qc_novel = [2, 1, 0] 
for k in range(1, 2**num_qpe_qubits):  
    lam, ratio = float(k), C / float(k)
    if ratio > 1.0: ratio = 1.0 
    angle = 2 * np.arcsin(ratio) 
    
    for i in range(num_qpe_qubits):
        if (k & (1 << i)) == 0: circ_inv_novel += X.on(qc_novel[i])
    circ_inv_novel += RY(angle).on(ancilla_qubit, qc_novel) 
    for i in range(num_qpe_qubits):
        if (k & (1 << i)) == 0: circ_inv_novel += X.on(qc_novel[i])

# ==================== 4. 终极测试引擎 ====================
def run_comparison_with_topology_noise(total_shots=1000000):
    results = {}
    test_cases = {
        "Standard_HHL": (circ_iqft_std, circ_inv_std),
        "Novel_HHL": (circ_iqft_novel, circ_inv_novel)
    }
    
    print("\n🚀 开始执行量子硬件仿真，计算原始解 x ......")
    for name, (iqft_module, inv_module) in test_cases.items():
        start_time = time.time()
        
        ideal_circ = circ_prep + circ_qpe + iqft_module + inv_module + iqft_module.hermitian() + circ_qpe.hermitian()
        noisy_circ = add_realistic_noise(ideal_circ, NOISE_CONFIG)
        
        noisy_circ += Measure('sol_q3').on(target_qubit) 
        noisy_circ += Measure('aux_q4').on(ancilla_qubit)
        
        sim = Simulator('mqvector', total_qubits)
        result = sim.sampling(noisy_circ, shots=total_shots)
        
        valid_counts = {k: v for k, v in result.data.items() if k.startswith('1')}
        total_valid_shots = sum(valid_counts.values())
        
        if total_valid_shots > 0:
            reconstructed_vector = np.zeros(2)
            for state, count in valid_counts.items():
                solution_state = state[1:]  
                prob = count / total_valid_shots
                reconstructed_vector[int(solution_state, 2)] = np.sqrt(prob)
            
            # 🌟 恢复原始解的绝对数值
            normalized_state = reconstructed_vector / np.linalg.norm(reconstructed_vector)
            raw_solution = normalized_state * TRUE_NORM
            
            # 🌟 计算态保真度 (Fidelity = Cosine_Similarity^2)
            ideal_state = np.array([2 / np.sqrt(5), 1 / np.sqrt(5)])
            fidelity = (np.dot(normalized_state, ideal_state))**2
            
            elapsed_time = time.time() - start_time
            
            results[name] = {
                'raw_solution': raw_solution,        # 原始方程解矢量
                'success_rate': total_valid_shots / total_shots,
                'fidelity': fidelity,                # 态保真度
                'execution_time': elapsed_time
            }
            print(f"✅ {name} 仿真完成耗时 {elapsed_time:.1f} 秒。")
            
    return results

# ==================== 5. 自动作图模块 (横向三联图) ====================
def plot_horizontal_results(results):
    print("\n📊 正在生成横向三联可视化报表...")
    
    labels = list(results.keys())
    survival_rates = [results[name]['success_rate'] for name in labels]
    fidelities = [results[name]['fidelity'] for name in labels]
    raw_solutions = [results[name]['raw_solution'] for name in labels]

    # 创建 1x3 的组合图，设置合适的宽度和高度
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 图 1：硬件生存率对比
    bars1 = axes[0].bar(labels, [s * 100 for s in survival_rates], color=['#4C72B0', '#DD8452'], width=0.5)
    axes[0].set_ylabel('Percentage (%)')
    axes[0].set_ylim(0, max(survival_rates)*100 + 20)
    for bar in bars1:
        yval = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, yval + 1.5, f'{yval:.2f}%', ha='center', va='bottom')

    # 图 2：态保真度对比
    bars2 = axes[1].bar(labels, [f * 100 for f in fidelities], color=['#55A868', '#C44E52'], width=0.5)
    axes[1].set_ylabel('Percentage (%)')
    axes[1].set_ylim(80, 105) 
    for bar in bars2:
        yval = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2, yval + 0.5, f'{yval:.2f}%', ha='center', va='bottom')

    # 图 3：原始解向量真实数值比对 (Ax = b)
    x = np.arange(2)  
    width = 0.25

    rects1 = axes[2].bar(x - width, TRUE_SOLUTION, width, label='理论真实解 x', color='#8C8C8C')
    rects2 = axes[2].bar(x, raw_solutions[0], width, label=labels[0], color='#4C72B0')
    rects3 = axes[2].bar(x + width, raw_solutions[1], width, label=labels[1], color='#DD8452')

    axes[2].set_xticks(x)
    axes[2].set_xticklabels(['分量 0 (基态)', '分量 1 (激发态)'])
    axes[2].set_ylabel('数值 (Value)')
    axes[2].set_ylim(0, max(TRUE_SOLUTION) + 0.5) # 给理论值上方留点空间
    axes[2].legend()

    # 添加数值标签
    def autolabel(rects, ax):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.4f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),  
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)

    autolabel(rects1, axes[2])
    autolabel(rects2, axes[2])
    autolabel(rects3, axes[2])

    plt.tight_layout()
    plt.subplots_adjust(top=0.85)
    plt.show()

# ==================== 6. 执行主程序 ====================
if __name__ == '__main__':
    # 按照你的设定运行 1,000,000 次 shot
    final_results = run_comparison_with_topology_noise(total_shots=1000000)
    plot_horizontal_results(final_results)

lamda 可二进制表示的量子系统 不同Qubit 误差对比


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from mindquantum.core.circuit import Circuit, UN
from mindquantum.core.gates import H, X, RZ, RY, Measure, PhaseShift, SWAP
from mindquantum.simulator import Simulator

# ==================== 1. 新型 IQFT 模块 ====================
def cz_and_swap(q_control, q_target, rot):
    c = Circuit()
    c += PhaseShift(np.pi * rot).on(q_target, q_control)
    c += SWAP.on([q_control, q_target])
    return c

def get_novel_iqft(num_qubits, qubits_list):
    c = Circuit()
    for i in range(num_qubits):
        c += H.on(qubits_list[0])
        for j in range(num_qubits - 1 - i):
            rot = 0.5 ** (j + 1)
            c += cz_and_swap(qubits_list[j], qubits_list[j+1], rot)
    return c.hermitian()

# ==================== 2. 动态时钟 HHL 求解器 (诱发泄露 + 量子滤波) ====================
# ==================== 2. 动态时钟 HHL 求解器 (诱发泄露 + 量子滤波) ====================
def run_4qubit_matrix_leakage_test(num_qpe_qubits, shots=500000):
    q_sol_1 = num_qpe_qubits
    q_sol_2 = num_qpe_qubits + 1
    q_aux   = num_qpe_qubits + 2
    total_qubits = num_qpe_qubits + 3
    
    # [模块 A] 状态制备 (|++>)
    circ_prep = Circuit()
    circ_prep += UN(H, [q_sol_1, q_sol_2])
    circ_prep += PhaseShift(np.pi).on(q_sol_2, q_sol_1)  # CZ = diag(1,1,1,-1)
    
    time_scale = 1
    
    # [模块 B] QPE (终极修复版：匹配 [高, 低, 高, 低] 模式)
    circ_qpe = Circuit()
    circ_qpe += UN(H, range(num_qpe_qubits))
    for k in range(num_qpe_qubits):
        tau = time_scale * np.pi * (2**(k - 1))
        
        # 1. 模拟 1.5 * I
        circ_qpe += PhaseShift(1.5 * tau).on(k)
        
        # 2. 【关键修正】：这里改为正号 (1.0 * tau)，相当于模拟 -0.5*X，
        # 完美反转特征值，让算法跑出 [高, 低, 高, 低] 的分布
        circ_qpe += H.on(q_sol_2)
        circ_qpe += RZ(1.0 * tau).on(q_sol_2, k) 
        circ_qpe += H.on(q_sol_2)

    # [模块 C] 新型 IQFT
    circ_iqft = get_novel_iqft(num_qpe_qubits, list(range(num_qpe_qubits))[::-1])

    # [模块 D] 穷举求逆与量子滤波
    circ_inv = Circuit()
    qc = list(range(num_qpe_qubits))
    C = 0.25 
    
    for k in range(1, 2**num_qpe_qubits):  
        lam = k / (time_scale * (2.0 ** (num_qpe_qubits - 2)))
        
        if lam < 0.5:
            continue
            
        ratio = min(C / lam, 1.0)
        angle = 2 * np.arcsin(ratio)
        
        for i in range(num_qpe_qubits):
            if (k & (1 << i)) == 0: circ_inv += X.on(qc[i])
        circ_inv += RY(angle).on(q_aux, qc)
        for i in range(num_qpe_qubits):
            if (k & (1 << i)) == 0: circ_inv += X.on(qc[i])

    # [模块 E] 组装与测量
    full_circ = circ_prep + circ_qpe + circ_iqft + circ_inv + circ_iqft.hermitian() + circ_qpe.hermitian()
    full_circ += Measure('sol_1').on(q_sol_1) 
    full_circ += Measure('sol_2').on(q_sol_2) 
    full_circ += Measure('aux').on(q_aux)

    # 运行模拟
    sim = Simulator('mqvector', total_qubits)
    res = sim.sampling(full_circ, shots=shots)
    
    valid_counts = {k: v for k, v in res.data.items() if k.startswith('1')}
    total_valid = sum(valid_counts.values())
    
    vec = np.zeros(4)
    if total_valid > 0:
        for state, count in valid_counts.items():
            vec[int(state[1:], 2)] = np.sqrt(count / total_valid)
            
        # 【关键修正】：删除了原来 unnorm_vec 的缩放计算，
        # 直接返回归一化态向量，使其高度能拔升到 0.63 和 0.31
        return vec
    return vec

# ==================== 3. 经典四柱绘图模块 ====================
def plot_qpe_convergence(theory_sol, res_q3, res_q4, res_q5):
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Songti SC', 'STHeiti', 'Arial'] 
    plt.rcParams['axes.unicode_minus'] = False
    
    labels = [r'$|00\rangle$', r'$|01\rangle$', r'$|10\rangle$', r'$|11\rangle$']
    x = np.arange(len(labels))
    width = 0.2  

    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    
    rects1 = ax.bar(x - 1.5*width, theory_sol, width, label='理论精确解', color='#2ca02c', edgecolor='white', zorder=3)
    rects2 = ax.bar(x - 0.5*width, res_q3, width, label='新型 HHL (时钟 3 Qubits)', color='#d62728', alpha=0.7, edgecolor='white', zorder=3)
    rects3 = ax.bar(x + 0.5*width, res_q4, width, label='新型 HHL (时钟 4 Qubits)', color='#ff7f0e', alpha=0.9, edgecolor='white', zorder=3)
    rects4 = ax.bar(x + 1.5*width, res_q5, width, label='新型 HHL (时钟 5 Qubits)', color='#1f77b4', hatch='//', edgecolor='white', zorder=3)

    ax.set_ylabel('解向量概率幅绝对值 (Amplitude)', fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=13)
    ax.set_ylim(0, 0.75) # 留出顶部空间显示数字
    
    ax.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
    ax.legend(fontsize=11, loc='upper left')

    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.4f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)

    autolabel(rects1)
    autolabel(rects2)
    autolabel(rects3)
    autolabel(rects4)

    fig.tight_layout()
    plt.show() 

# ==================== 4. 主程序入口 ====================
if __name__ == "__main__":
    test_shots = 1000000 
    
    # 经典的理论解：对应归一化后的 [0.25, 0.50, 0.25, 0.50]
    theory_sol = [np.sqrt(2/5), np.sqrt(1/10), np.sqrt(2/5), np.sqrt(1/10)]
    
    print("正在运行 3-Qubit 时钟 HHL... ")
    res_q3 = run_4qubit_matrix_leakage_test(num_qpe_qubits=3, shots=test_shots)
    
    print("正在运行 4-Qubit 时钟 HHL... ")
    res_q4 = run_4qubit_matrix_leakage_test(num_qpe_qubits=4, shots=test_shots)
    
    print("正在运行 5-Qubit 时钟 HHL... ")
    res_q5 = run_4qubit_matrix_leakage_test(num_qpe_qubits=5, shots=test_shots)
    
    print("\n绘制收敛对比图...")
    plot_qpe_convergence(theory_sol, res_q3, res_q4, res_q5)

lamda 不可二进制表示 不同Qubit 误差对比


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from mindquantum.core.circuit import Circuit, UN
from mindquantum.core.gates import H, X, RZ, RY, Measure, PhaseShift, SWAP
from mindquantum.simulator import Simulator

# ==================== 1. 新型 IQFT 模块 ====================
def cz_and_swap(q_control, q_target, rot):
    c = Circuit()
    c += PhaseShift(np.pi * rot).on(q_target, q_control)
    c += SWAP.on([q_control, q_target])
    return c

def get_novel_iqft(num_qubits, qubits_list):
    c = Circuit()
    for i in range(num_qubits):
        c += H.on(qubits_list[0])
        for j in range(num_qubits - 1 - i):
            rot = 0.5 ** (j + 1)
            c += cz_and_swap(qubits_list[j], qubits_list[j+1], rot)
    return c.hermitian()

# ==================== 2. 动态时钟 HHL 求解器 (自然泄露矩阵 + 量子滤波) ====================
def run_natural_leakage_test(num_qpe_qubits, shots=500000):
    # 目标矩阵: A = 1.2I + 0.4 X⊗Z 
    # 特征值: 1.6 和 0.8 (其对应的 QPE 相位 0.8 和 0.4 为二进制无限循环小数)
    # 理论解: [0.3125, 0.6250, 0.3125, 0.6250]
    
    q_sol_1 = num_qpe_qubits
    q_sol_2 = num_qpe_qubits + 1
    q_aux   = num_qpe_qubits + 2
    total_qubits = num_qpe_qubits + 3
    
    # [模块 A] 状态制备 (|++>)
    circ_prep = Circuit()
    circ_prep += UN(H, [q_sol_1, q_sol_2])
    
    # [模块 B] QPE (无任何人工干预，自然演化)
    circ_qpe = Circuit()
    circ_qpe += UN(H, range(num_qpe_qubits))
    for k in range(num_qpe_qubits):
        tau = np.pi * (2**(k - 1))
        
        # 矩阵 A = 1.2I + 0.4 X⊗Z
        circ_qpe += PhaseShift(1.2 * tau).on(k)
        circ_qpe += H.on(q_sol_2)        
        circ_qpe += X.on(q_sol_2, q_sol_1)     
        circ_qpe += RZ(-0.8 * tau).on(q_sol_2, k)  # 0.4 * 2 = 0.8
        circ_qpe += X.on(q_sol_2, q_sol_1)     
        circ_qpe += H.on(q_sol_2)

    # [模块 C] 新型 IQFT
    circ_iqft = get_novel_iqft(num_qpe_qubits, list(range(num_qpe_qubits))[::-1])
# [模块 D] 穷举求逆与量子滤波
    circ_inv = Circuit()
    qc = list(range(num_qpe_qubits))
    C = 0.4 
    
    for k in range(1, 2**num_qpe_qubits):  
        # 【终极修复 1】：正确的相位解码公式 (除以 2^(n-2))
        lam = k / (2.0 ** (num_qpe_qubits - 2))
        
        # 【终极修复 2】：特征值截断滤波
        # 真实最小特征值为 0.8，我们过滤掉由狄利克雷泄露产生的低于 0.4 的纯噪声
        if lam < 0.4:
            continue
            
        ratio = min(C / lam, 1.0)
        angle = 2 * np.arcsin(ratio)
        
        for i in range(num_qpe_qubits):
            if (k & (1 << i)) == 0: circ_inv += X.on(qc[i])
        circ_inv += RY(angle).on(q_aux, qc)
        for i in range(num_qpe_qubits):
            if (k & (1 << i)) == 0: circ_inv += X.on(qc[i])

    # [模块 E] 组装与测量
    full_circ = circ_prep + circ_qpe + circ_iqft + circ_inv + circ_iqft.hermitian() + circ_qpe.hermitian()
    full_circ += Measure('sol_1').on(q_sol_1) 
    full_circ += Measure('sol_2').on(q_sol_2) 
    full_circ += Measure('aux').on(q_aux)

    # 运行模拟
    sim = Simulator('mqvector', total_qubits)
    res = sim.sampling(full_circ, shots=shots)
    
    valid_counts = {k: v for k, v in res.data.items() if k.startswith('1')}
    total_valid = sum(valid_counts.values())
    
    vec = np.zeros(4)
    if total_valid > 0:
        for state, count in valid_counts.items():
            vec[int(state[1:], 2)] = np.sqrt(count / total_valid)
            
        # 还原未归一化的绝对幅值 
        unnorm_vec = vec * (np.sqrt(total_valid / shots) / C)
        return unnorm_vec
    return vec

# ==================== 3. 经典四柱绘图模块 ====================
def plot_qpe_convergence(theory_sol, res_q3, res_q4, res_q5):
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Songti SC', 'STHeiti', 'Arial'] 
    plt.rcParams['axes.unicode_minus'] = False
    
    labels = [r'$|00\rangle$', r'$|01\rangle$', r'$|10\rangle$', r'$|11\rangle$']
    x = np.arange(len(labels))
    width = 0.2  

    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    
    rects1 = ax.bar(x - 1.5*width, theory_sol, width, label='理论精确解', color='#2ca02c', edgecolor='white', zorder=3)
    rects2 = ax.bar(x - 0.5*width, res_q3, width, label='新型 HHL (时钟 3 Qubits)', color='#d62728', alpha=0.7, edgecolor='white', zorder=3)
    rects3 = ax.bar(x + 0.5*width, res_q4, width, label='新型 HHL (时钟 4 Qubits)', color='#ff7f0e', alpha=0.9, edgecolor='white', zorder=3)
    rects4 = ax.bar(x + 1.5*width, res_q5, width, label='新型 HHL (时钟 5 Qubits)', color='#1f77b4', hatch='//', edgecolor='white', zorder=3)

    ax.set_ylabel('解向量概率幅绝对值 (Amplitude)', fontsize=13)
    
    # 标题更新：强调是由无限循环小数引发的自然泄露
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=13)
    ax.set_ylim(0, 0.8) # 留出顶部空间显示 0.6250
    
    ax.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
    ax.legend(fontsize=11, loc='upper left')

    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.4f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)

    autolabel(rects1)
    autolabel(rects2)
    autolabel(rects3)
    autolabel(rects4)

    fig.tight_layout()
    plt.show() 

# ==================== 4. 主程序入口 ====================
if __name__ == "__main__":
    test_shots = 1000000 
    
    # 针对新矩阵的精准理论解
    theory_sol = [0.3125, 0.6250, 0.3125, 0.6250]
    
    print("正在运行 3-Qubit 时钟 HHL... (严重泄露)")
    res_q3 = run_natural_leakage_test(num_qpe_qubits=3, shots=test_shots)
    
    print("正在运行 4-Qubit 时钟 HHL... (泄露减轻)")
    res_q4 = run_natural_leakage_test(num_qpe_qubits=4, shots=test_shots)
    
    print("正在运行 5-Qubit 时钟 HHL... (收敛至理论解)")
    res_q5 = run_natural_leakage_test(num_qpe_qubits=5, shots=test_shots)
    
    print("\n绘制收敛对比图...")
    plot_qpe_convergence(theory_sol, res_q3, res_q4, res_q5)

shots 带来的误差


In [ ]:
import time
import numpy as np

from mindquantum.core.circuit import Circuit, UN
from mindquantum.core.gates import H, X, RY, RX, Measure, PhaseShift, SWAP
from mindquantum.simulator import Simulator
import matplotlib
import matplotlib.font_manager as fm
import shutil
import os
import matplotlib.ticker as ticker
try:
    fm.fontManager.__init__()  # 重新初始化
except:
    pass
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Songti SC', 'STHeiti', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['mathtext.fontset'] = 'stixsans'  # 关键：使用 STIX 替代 cm

import matplotlib.pyplot as plt  # 在这之后导入

# ==================== 1. 参数设置 ====================
num_qpe_qubits = 2       
num_solution_qubits = 1  
total_qubits = 4         
C = 1.0

# ==================== 2. 严格使用你的原版代码模块 ====================
# [模块 A] 状态制备
circ_prep = Circuit()

# [模块 B] 量子相位估算 (QPE) - 正向演化
circ_qpe = Circuit()
circ_qpe += UN(H, range(num_qpe_qubits)) 
for k in range(num_qpe_qubits):
    tau = np.pi * (2**(k - 1))  
    circ_qpe += PhaseShift(1.5 * tau).on(k)
    circ_qpe += RX(-tau).on(2, k)

# [模块 D] 穷举求逆 (汉堡包结构)
circ_inv = Circuit()
qc = [0, 1]  
qs = 3       
for k in range(1, 2**num_qpe_qubits):  
    lam = float(k)
    ratio = C / lam
    if ratio > 1.0: ratio = 1.0 
    angle = 2 * np.arcsin(ratio)
    for i in range(num_qpe_qubits):
        if (k & (1 << i)) == 0:
            circ_inv += X.on(qc[i])
    circ_inv += RY(angle).on(qs, qc)
    for i in range(num_qpe_qubits):
        if (k & (1 << i)) == 0:
            circ_inv += X.on(qc[i])

# [模块 C2] 本文引入的新型近邻 IQFT (完全使用你的严谨实现)
def cz_and_swap(q_control, q_target, rot):
    c = Circuit()
    c += PhaseShift(np.pi * rot).on(q_target, q_control)
    c += SWAP.on([q_control, q_target])
    return c

def get_novel_iqft(num_qubits, qubits_list):
    c = Circuit()
    for i in range(num_qubits):
        c += H.on(qubits_list[0])
        for j in range(num_qubits - 1 - i):
            rot = 0.5 ** (j + 1)
            c += cz_and_swap(qubits_list[j], qubits_list[j+1], rot)
    # 返回其共轭转置，即 IQFT
    return c.hermitian()

circ_iqft_novel = get_novel_iqft(2, [1, 0])

# 完美对称的线路组装
full_circ = circ_prep + circ_qpe + circ_iqft_novel + circ_inv + circ_iqft_novel.hermitian() + circ_qpe.hermitian()
full_circ += Measure('sol_q2').on(2) 
full_circ += Measure('aux_q3').on(3)

# ==================== 3. 核心实验：不同 Shots 下的散粒噪声 ====================
# 设置采样梯度 (从 100 到 10,000,000)
shots_list = [100, 500, 1000, 5000, 10000, 50000, 100000, 500000, 1000000, 5000000, 10000000]
exact_solution = np.array([0.75, 0.25])
experimental_errors = []

print("开始散粒噪声 (Shot Noise) 实验分析...")
sim = Simulator('mqvector', total_qubits)

# 多次实验取平均以稳定小 Shots 下的波动
num_trials = 5 

for shots in shots_list:
    current_shot_errors = []
    for _ in range(num_trials):
        result = sim.sampling(full_circ, shots=shots)
        valid_counts = {k: v for k, v in result.data.items() if k.startswith('1')}
        total_valid = sum(valid_counts.values())
        
        if total_valid > 0:
            vec = np.zeros(2)
            for state, count in valid_counts.items():
                vec[int(state[1:], 2)] = np.sqrt(count / total_valid)
            vec = vec / np.linalg.norm(vec)
            final_sol = vec * ((np.sqrt(total_valid / shots) * 1.0) / C)
            
            # 计算绝对误差 (L2范数)
            error = np.linalg.norm(final_sol - exact_solution)
            current_shot_errors.append(error)
        else:
            current_shot_errors.append(1.0) # 失败分支
            
    avg_error = np.mean(current_shot_errors)
    experimental_errors.append(avg_error)
    print(f"Shots: {shots:<10} | 平均解向量 L2 误差: {avg_error:.6f}")

# ==================== 4. 学术绘图 (双对数坐标系 Log-Log) ====================
# ==================== 4. 学术绘图 (双对数坐标系 Log-Log) ====================
print("\n正在生成双对数坐标系下的散粒噪声拟合图表...")

# 计算理论参考曲线 O(1/sqrt(N))
base_error = experimental_errors[0]
base_shot = shots_list[0]
theoretical_errors = [base_error * np.sqrt(base_shot) / np.sqrt(s) for s in shots_list]

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

# 画实验数据点 (蓝色实线)
ax.loglog(shots_list, experimental_errors, marker='o', markersize=8, linestyle='-', linewidth=2, 
          color='#1f77b4', label='新型 HHL 解向量误差')

# 画理论基准线 (红色虚线)
ax.loglog(shots_list, theoretical_errors, marker='', linestyle='--', linewidth=2, 
          color='#d62728', label=r'理论散粒噪声收敛极限 $\mathcal{O}(1/\sqrt{N_{shots}})$')

# ----------------- 新增部分：修改 Y 轴刻度格式 -----------------
formatter = ticker.ScalarFormatter()
formatter.set_scientific(False)  # 禁用科学计数法
ax.yaxis.set_major_formatter(formatter)
# -------------------------------------------------------------

ax.set_xlabel('有限次采样规模 $N_{shots}$ (Log Scale)', fontsize=14)
ax.set_ylabel('解向量绝对误差 (Log Scale)', fontsize=14)

ax.legend(fontsize=12)

plt.tight_layout()
plt.show()